[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day8_solution.ipynb)

# Day 8 · 정답 — LLM 과 프롬프트

모델을 직접 열어 토큰 · 확률 · 어텐션을 본다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

`live` 와 `lab` 의 모든 문제에 대한 정답본이다.
수강생은 먼저 스스로 풀어 본 뒤에 연다.

두 벌을 합쳐 담으므로 **문제 번호가 `lab` 과 다르다.** 번호 대신
**지문으로 찾는다.**

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 모델을 연다

In [ ]:
# 1) 오늘 쓸 것들을 불러온다
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModelForCausalLM
print('torch', torch.__version__)

In [ ]:
# 2) 모델을 받아 온다 — 계수 15억 개짜리 한 대
NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
tok = AutoTokenizer.from_pretrained(NAME)
# 어텐션을 꺼내 보려면 attn_implementation='eager' 여야 한다
model = AutoModelForCausalLM.from_pretrained(NAME, attn_implementation='eager')
model.eval()
print('계수 %.1f억 개' % (sum(p.numel() for p in model.parameters()) / 1e8))

In [ ]:
# 3) 장치로 옮긴다 — cpu 면 느리지만 돌기는 한다
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print('장치', device)

## 2. 토큰 — 모델이 실제로 받는 단위

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 1.** 아래 문장이 몇 토큰인지 세어 `n` 에 담는다.

In [ ]:
q = '오늘 설비 점검에서 이상이 발견되었다'
n = len(tok(q)['input_ids'])

print(n)
assert isinstance(n, int) and n > 0

## 3. 임베딩 — 토큰이 숫자 줄이 된다

In [ ]:
# 토큰 하나가 몇 칸짜리 숫자 줄인지 본다
E = model.get_input_embeddings().weight
print('어휘 %d개 · 한 토큰 %d칸' % (len(tok), E.shape[1]))
print('표는 %d줄 — 계산이 빠른 크기로 맞춰 두느라 어휘보다 조금 길다' % E.shape[0])

In [ ]:
# 낱말 하나의 숫자 줄을 꺼내는 함수
def vec(w):
    return E[tok(w, add_special_tokens=False)['input_ids'][0]]

print('king 의 숫자 줄', tuple(vec(' king').shape))

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 2.** 우리말 낱말 세 쌍을 골라 가까운 정도를 재 본다.
> 뜻이 가까운 쌍과 먼 쌍이 숫자로 갈리는지 본다.

In [ ]:
for a, b in ((' 왕', ' 여왕'), (' 왕', ' 바나나'), (' 고양이', ' 개')):
    print('%-8s ~%-8s  %.3f' % (a, b, F.cosine_similarity(vec(a), vec(b), dim=0).item()))
print('영어보다 덜 갈릴 수 있다 — 토큰이 쪼개져서다')

## 4. 다음 한 토큰

In [ ]:
# 앞부분을 넣고 다음 토큰의 점수를 받아 온다
head = '대한민국의 수도는'
x = tok(head, return_tensors='pt').to(device)
with torch.no_grad():
    logits = model(**x).logits[0, -1]
probs = logits.softmax(-1)
print('후보 %d개에 확률이 매겨졌다' % probs.shape[0])

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 3.** 1등 토큰의 확률을 `p1` 에 담는다.

In [ ]:
p1 = probs.max().item()

print('%.4f' % p1)
assert 0 < p1 <= 1

## 5. 온도 — 차이를 얼마나 벌릴지

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 4.** 온도를 **0.1 부터 3.0 까지** 옮기며 1등 확률이 어떻게 되는지 그래프로 그린다.
> 낮추면 1로 붙고 높이면 평평해지는 것이 보여야 한다.

In [ ]:
ts = [0.1, 0.3, 0.5, 0.8, 1.0, 1.5, 2.0, 3.0]
ys = [(logits / t).softmax(-1).max().item() for t in ts]
plt.plot(ts, ys, marker='o'); plt.xlabel('temperature'); plt.ylabel('1등 확률')
plt.grid(alpha=.3); plt.show()
print('온도는 순위를 바꾸지 않는다 — 차이를 얼마나 크게 볼지만 정한다')

## 6. 어텐션 — 어디를 보고 고르나

In [ ]:
# 어텐션 가중치를 같이 받아 온다
s2 = 'The bank of the river was very steep'
x2 = tok(s2, return_tensors='pt').to(device)
with torch.no_grad():
    out = model(**x2, output_attentions=True)
A = out.attentions
print('층 %d개 · 층마다 머리 %d개 · 토큰 %d개' % (len(A), A[0].shape[1], A[0].shape[-1]))

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **실습문제 5.** 층과 머리를 바꿔 가며 그려 본다.
> 앞 층은 옆 토큰을, 뒤 층은 멀리 있는 토큰을 보는 경향이 있는지 확인한다.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
for a, (L, H) in zip(ax, [(0, 0), (14, 0), (26, 3)]):
    a.imshow(A[L][0, H].float().cpu(), cmap='Purples')
    a.set_title('%d층 %d번' % (L, H)); a.set_xticks([]); a.set_yticks([])
plt.show()
print('머리마다 보는 자리가 다르다')

## 7. 프롬프트는 앞부분이다

In [ ]:
# 앞으로 쓸 답 만들기 함수 — 채팅 틀을 씌워 이어 쓰게 한다
def gen(user, system=None, n=56):
    ms = ([{'role': 'system', 'content': system}] if system else []) \
         + [{'role': 'user', 'content': user}]
    p = tok.apply_chat_template(ms, tokenize=False, add_generation_prompt=True)
    x = tok(p, return_tensors='pt').to(device)
    with torch.no_grad():
        y = model.generate(**x, max_new_tokens=n, do_sample=False)
    return tok.decode(y[0][x['input_ids'].shape[1]:], skip_special_tokens=True)

In [ ]:
# 오늘 물어볼 한 문장
ask = '설비 점검에서 소음이 크고 진동이 있다.'
print(ask)

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 6.** 칸 이름을 하나 더 넣어(`재발 방지:`) 다시 돌려 `out` 에 담는다.

In [ ]:
form2 = ask + '\n아래 형식으로만 답하라.\n증상: \n의심 원인: \n조치: \n재발 방지: '
out = gen(form2, n=120)

print(out)
assert '재발' in out

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 7.** 현장에서 쓰는 설비 이름 세 개를 물어보고 답이 맞는지 본다.
> 틀린 답을 얼마나 자신 있게 말하는지 함께 확인한다.

In [ ]:
for w in ('고로', '전로', '소성로'):
    print('[%s] %s\n' % (w, gen('%s가 뭐야? 한 문장으로 답해라.' % w, n=40)))
print('모르면 비운 채 두지 않고 그럴듯한 말을 채워 넣는다')